# Module 00 Lab — Build a Bare-Metal Agent

**Course:** AI Agents Teaching Platform  
**Module:** 00 — Mental Models  
**Estimated time:** 2–3 hours

---

### What you'll build

A working agent loop **from scratch** — no LangChain, no agent frameworks, no abstractions beyond a single API library. Just Python.

By the end you'll have:
- A safe calculator tool
- A ~50-line agent loop that calls the LLM, executes tools, and terminates correctly

### Constraints
- No LangChain, LangGraph, AutoGen, or similar frameworks
- Loop must have a working termination condition **and** a `MAX_STEPS` guardrail

---

> **VS Code / local?** See the lab page on the platform for instructions.

## Step 1 — Choose your model

Set `MODEL` to any model string supported by [LiteLLM](https://docs.litellm.ai/docs/providers). Common options:

| Provider | Example model string | Key env var |
|---|---|---|
| Anthropic | `claude-haiku-4-5-20251001` | `ANTHROPIC_API_KEY` |
| OpenAI | `gpt-4o-mini` | `OPENAI_API_KEY` |
| Google Gemini | `gemini/gemini-1.5-flash` | `GEMINI_API_KEY` |
| Mistral | `mistral/mistral-small-latest` | `MISTRAL_API_KEY` |
| Groq | `groq/llama-3.1-8b-instant` | `GROQ_API_KEY` |
| Ollama (local) | `ollama/llama3` | *(no key needed)* |

All are cheap or free. The full provider list is at [docs.litellm.ai/docs/providers](https://docs.litellm.ai/docs/providers).

---

> **Why does it not matter which model you pick?**  
> This is the point of the lab. The agent loop — the `for` loop, the tool registry, the message list, the termination check — is **identical regardless of provider**. Only the model string changes. The LLM is one swappable component; everything else is yours.

In [ ]:
# ── Set your model string here ─────────────────────────────
MODEL = "claude-haiku-4-5-20251001"
# ──────────────────────────────────────────────────────────

print(f"Model: {MODEL}")

## Step 2 — Install dependencies

In [ ]:
%pip install -q litellm

## Step 3 — API key

Set the environment variable that matches your provider (see the table above). It stays in this session only.

If you're using Ollama locally, skip this cell — no key is needed.

In [ ]:
import os
from getpass import getpass

# Change the variable name to match your provider
os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your API key: ")
print("Key set ✓")

## Step 4 — Build the calculator tool

The agent needs at least one tool. We'll build a **safe calculator** that uses Python's AST parser instead of `eval()`. This matters because the expression comes from the model — untrusted input.

### Why not `eval(expression)`?

```python
eval("__import__('os').system('rm -rf /')")
```

`eval` would execute that. The AST approach only allows numbers and four arithmetic operators — anything else returns an error string.

### Your TODO

`_safe_eval` is complete. Fill in `calculator()` so it:
1. Parses the expression string into an AST
2. Calls `_safe_eval` on the root node
3. Returns the result as a string
4. Catches any exception and returns a descriptive error string — never crashes

In [ ]:
import ast
import operator as op

_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    raise ValueError(f"Unsupported expression: {ast.dump(node)}")


def calculator(args: dict) -> str:
    """Safe calculator. args must contain 'expression' key."""
    # TODO: implement
    # Hint: ast.parse(expr, mode="eval").body gives you the root node
    pass


TOOLS = {"calculator": calculator}


# Self-test
assert calculator({"expression": "2 + 2"}) == "4",          "Basic addition failed"
assert calculator({"expression": "10 / 2"}) == "5.0",       "Division failed"
assert "error" in calculator({"expression": "__import__('os')"}).lower(), "Should return error for unsafe input"
print("calculator() tests passed ✓")

<details>
<summary>💡 Stuck? Reveal the calculator solution</summary>

```python
def calculator(args: dict) -> str:
    try:
        tree = ast.parse(args["expression"], mode="eval")
        result = _safe_eval(tree.body)
        return str(result)
    except Exception as e:
        return f"calculator error: {e}"
```

</details>

## Step 5 — Build the agent loop

The loop:
1. Sends the goal (plus conversation history) to the LLM
2. The LLM replies with JSON — either a tool call or a final answer
3. If a tool call, execute it and append the result
4. Repeat until done or `MAX_STEPS` is reached

### JSON protocol

**Tool call:**
```json
{"thought": "I need to compute 24 × 7", "action": "calculator", "args": {"expression": "24 * 7"}}
```
**Final answer:**
```json
{"thought": "I have the answer", "action": "final", "answer": "168"}
```

### LiteLLM message format

LiteLLM uses the OpenAI message format, which every major provider accepts:

```python
import litellm

response = litellm.completion(
    model=MODEL,
    messages=[
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": "What is 2 + 2?"},
        {"role": "assistant", "content": '{"action": "calculator", ...}'},
        {"role": "user",      "content": "Observation: 4"},
    ]
)
text = response.choices[0].message.content
```

That's it — same call regardless of whether `MODEL` points at Anthropic, OpenAI, Gemini, or anything else.

### The five TODOs

In [ ]:
import json
import logging
import litellm

litellm.drop_params = True  # silently ignore unsupported params across providers

logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger(__name__)

MAX_STEPS = 5

SYSTEM_PROMPT = """You are a small tool-using agent. Always reply with valid JSON — nothing else.

To use a tool:
  {"thought": "<your reasoning>", "action": "calculator", "args": {"expression": "<math expression>"}}

To give a final answer:
  {"thought": "<your reasoning>", "action": "final", "answer": "<your answer>"}
"""


def build_initial_messages(goal: str) -> list[dict]:
    """TODO 1: Return the starting message list.

    Include the system prompt as the first message, then the user goal:
      [{"role": "system", "content": SYSTEM_PROMPT},
       {"role": "user",   "content": goal}]
    """
    # TODO 1
    pass


def call_model(messages: list[dict]) -> str:
    """TODO 2: Call the LLM via LiteLLM and return its text response.

    Use litellm.completion(model=MODEL, max_tokens=256, messages=messages)
    and return response.choices[0].message.content
    """
    # TODO 2
    pass


def parse_action(text: str) -> dict:
    """TODO 3: Parse the model's JSON response.

    If invalid JSON or missing 'action' key, return a safe default:
      {"action": "final", "answer": "<error description>"}
    so the loop always terminates cleanly.
    """
    # TODO 3
    pass


def run_agent(goal: str) -> str:
    messages = build_initial_messages(goal)

    for step in range(MAX_STEPS):
        raw = call_model(messages)
        action = parse_action(raw)
        log.info("step %d | action: %s", step + 1, action.get("action"))

        # TODO 4: if action["action"] == "final", return action["answer"]

        # TODO 5: otherwise look up the tool in TOOLS, call it with action["args"],
        #         append the assistant turn and the observation to messages, then continue
        pass

    return "max steps reached — try a simpler goal"

<details>
<summary>💡 Stuck? Reveal the full agent solution</summary>

```python
def build_initial_messages(goal: str) -> list[dict]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": goal},
    ]


def call_model(messages: list[dict]) -> str:
    response = litellm.completion(model=MODEL, max_tokens=256, messages=messages)
    return response.choices[0].message.content


def parse_action(text: str) -> dict:
    try:
        action = json.loads(text)
    except json.JSONDecodeError:
        return {"action": "final", "answer": "Model returned invalid JSON — stopping safely."}
    if "action" not in action:
        return {"action": "final", "answer": "No action key in model output — stopping safely."}
    return action


def run_agent(goal: str) -> str:
    messages = build_initial_messages(goal)
    for step in range(MAX_STEPS):
        raw = call_model(messages)
        action = parse_action(raw)
        log.info("step %d | action: %s", step + 1, action.get("action"))

        # TODO 4
        if action["action"] == "final":
            return action["answer"]

        # TODO 5
        tool_name = action["action"]
        observation = (
            TOOLS[tool_name](action.get("args", {}))
            if tool_name in TOOLS
            else f"Unknown tool: {tool_name}"
        )
        messages.append({"role": "assistant", "content": raw})
        messages.append({"role": "user",      "content": f"Observation: {observation}"})

    return "max steps reached — try a simpler goal"
```

Notice: `call_model` is **two lines**. It doesn't know or care which provider `MODEL` points at.

</details>

## Step 6 — Test your agent

In [ ]:
# Test 1: basic arithmetic
result = run_agent("What is 24 multiplied by 7?")
print("Result:", result)
assert "168" in result, f"Expected 168 in result, got: {result}"
print("Test 1 passed ✓")

In [ ]:
# Test 2: multi-step — needs two tool calls
result = run_agent("What is (15 + 27) multiplied by 4?")
print("Result:", result)
assert "168" in result, f"Expected 168 in result, got: {result}"
print("Test 2 passed ✓")

In [ ]:
# Test 3: unknown tool — must not crash
result = run_agent("Search the web for the capital of France.")
print("Result:", result)
print("Test 3 passed ✓ (didn't crash)")

## Step 7 — Experiments

### Experiment A: break the termination
Change `MAX_STEPS = 5` to `MAX_STEPS = 1` and run a multi-step goal. What happens?

### Experiment B: remove the JSON guard
In `parse_action`, remove the `try/except` and replace with `return json.loads(text)`. Run a conversational goal (e.g. "Tell me a joke"). What error do you get? Why does the guard exist?

### Experiment C: temperature
Add `temperature=1.0` to `litellm.completion(...)` and run the same arithmetic goal five times. Then try `temperature=0.0`. Do you see different behaviour?

### Experiment D: swap the model
Change `MODEL` in Step 1 to a different provider (e.g. `gpt-4o-mini`, `gemini/gemini-1.5-flash`), set the matching API key in Step 3, then re-run from Step 5 down. Do all three tests still pass? Which function did you have to change? (Answer: none — only the `MODEL` string.)

In [ ]:
# Scratch cell — use this for experiments


## Step 8 — Reflection

1. **What does your agent do?**
2. **What breaks if you remove the loop?**
3. **What changes at `temperature=1.0` vs `0.0`?**
4. **In Experiment D, what was the only thing you changed to switch providers?**

*(Double-click to edit)*

1. 
2. 
3. 
4. 

## Done!

Head back to the platform and take the **Module Quiz** to mark Module 00 complete.

**Rubric reminder** — Meets (3) on all five criteria:
- Loop runs and terminates correctly
- You can explain every line cold
- You can draw and explain the agent loop correctly
- You can name a problem where a plain LLM call beats an agent loop
- README covers what breaks without the loop and temperature effects